In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

### 사칙 연산 툴

In [4]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    두 숫자를 더해 그 합을 반환합니다.

    Args:
        a: 더할 첫 번째 숫자
        b: 더할 두 번째 숫자
    """
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """
    첫 번째 숫자에서 두 번째 숫자를 뺍니다.

    Args:
        a: 빼기 연산의 대상이 되는 수 (피감수)
        b: 뺄 숫자 (감수)
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """
    두 숫자를 곱해 그 결과를 반환합니다.

    Args:
        a: 곱할 첫 번째 숫자
        b: 곱할 두 번째 숫자
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """
    첫 번째 숫자를 두 번째 숫자로 나눕니다. 
    0으로 나누려고 하면 에러 메시지를 반환합니다.

    Args:
        a: 나뉘는 수 (피제수)
        b: 나누는 수 (제수)
    """
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return a / b

# 툴 리스트
tools = [add, subtract, multiply, divide]

In [11]:
print(f"더하기 테스트: {add.invoke({'a': 10, 'b': 5})}")
print(f"나누기 테스트: {divide.invoke({'a': 10, 'b': 2})}")

더하기 테스트: 15
나누기 테스트: 5.0


In [7]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    temperature=0,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
agent = create_agent(model, tools)

In [9]:
# 도구 호출 하지 않을 수도 있음
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [10]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='b784aa57-09d7-40d4-9208-4595923ccc84'),
  AIMessage(content='계산 순서에 따라 123 + (456 * 789) / 12를 계산해야 합니다.\n\n먼저 456 * 789를 계산하겠습니다.', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 789, "a": 456}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba1c2-df1d-7ab0-bd1d-c11b110ecdd5-0', tool_calls=[{'name': 'multiply', 'args': {'b': 789, 'a': 456}, 'id': 'dbc6c5b8-ad2f-44df-ba2b-bd72fc2de9b5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 356, 'output_tokens': 69, 'total_tokens': 425, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='359784', name='multiply', id='59cf2ac3-5817-46ad-827e-4c5ecaaa29fd', tool_call_id='dbc6c5b8-ad2f-44df-ba2b-bd72fc2de9b5'),
  AIMessage(content='', additional_kwargs

In [62]:
agent = create_agent(
    model, 
    tools,
    system_prompt="당신은 유능하고 친절한 수학 선생님입니다. 사칙 연산 요청 시 계산 우선 순위별로 단계를 나누고 계획을 세웁니다. 순서대로 Tool을 이용해 계산한 후 최종 답변을 생성합니다.",
)

In [63]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [64]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='516422b5-2874-48ca-82df-0d2ca950dfe6'),
  AIMessage(content='계산 순서에 따라 문제를 해결해 드릴게요.\n\n1. 곱셈: 456 * 789\n2. 나눗셈: (1단계 결과) / 12\n3. 덧셈: 123 + (2단계 결과)\n\n먼저 곱셈부터 계산해 보겠습니다.', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 456, "b": 789}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b9be8-899f-7a30-b3f5-a7cf1dbfbfc0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 456, 'b': 789}, 'id': '2be1cea9-eb8f-49c6-b75d-1e987b803b9d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 407, 'output_tokens': 93, 'total_tokens': 500, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='359784', name='multiply', id='c4a1bf87-9ab4-4cb7-84ee-59828afeb406', tool_call_id='2be1cea9-eb8f-49c6-b75d-1e987b803b9d'),